# Publication copy

Outputs and machine-specific paths were removed. Run `python scripts/prepare_local_artifacts.py` first. GPU experiments additionally require datasets, authorized pretrained models and the large artifacts listed in `docs/LARGE_ARTIFACTS.md`. Do not overwrite the frozen reporting inputs.


In [ ]:
from pathlib import Path
import os, sys
_start = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).expanduser().resolve()
_PUBLICATION_ROOT = next((p for p in (_start, *_start.parents)
                         if (p / "Methods").is_dir() and (p / "README.md").is_file()), None)
if _PUBLICATION_ROOT is None:
    raise FileNotFoundError("Set PROJECT_ROOT to the cloned cancer_image_pathology folder")
os.chdir(_PUBLICATION_ROOT)
sys.path.insert(0, str(_PUBLICATION_ROOT))
os.environ["PROJECT_ROOT"] = str(_PUBLICATION_ROOT)
print("Project:", _PUBLICATION_ROOT)


# UNI intervention-based patch attribution

**Research question:** Does intervention on UNI patch representations produce more faithful patch-level explanations than gradient-weighted Transformer attribution for colorectal histology classification?

This experiment reuses the frozen cohort, grouped OOF folds, UNI encoder, classifier heads, preprocessing, and faithfulness protocol from `04_grouped_oof_faithfulness.ipynb`. It does not train another classifier and does not add a causal-ranking loss.

Here, "causal" means an intervention effect on the model's target-class score. It does not imply that a patch is biologically causal for cancer or tissue pathology.


In [ ]:
from pathlib import Path
import hashlib
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
def locate_project_root():
    return _PUBLICATION_ROOT


PROJECT_ROOT = locate_project_root()
DATASET_DIR = (
    PROJECT_ROOT
    / 'Colorectal Histology MNIST'
    / 'Kather_texture_2016_image_tiles_5000'
    / 'Kather_texture_2016_image_tiles_5000'
)
SOURCE_ARTIFACT_DIR = PROJECT_ROOT / 'artifacts' / 'grouped_oof_faithfulness'
UNI_CHECKPOINT_DIR = SOURCE_ARTIFACT_DIR / 'checkpoints' / 'uni'
COHORT_PATH = SOURCE_ARTIFACT_DIR / 'faithfulness_cohort_manifest.csv'
COHORT_METADATA_PATH = SOURCE_ARTIFACT_DIR / 'faithfulness_cohort_manifest.metadata.json'
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts' / 'uni_intervention_causal'
FAITHFULNESS_DIR = ARTIFACT_DIR / 'faithfulness'
STABILITY_DIR = ARTIFACT_DIR / 'stability'
FIGURE_DIR = ARTIFACT_DIR / 'figures'
for directory in (FAITHFULNESS_DIR, STABILITY_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Project:', PROJECT_ROOT)
print('Source cohort:', COHORT_PATH)
print('New artifacts:', ARTIFACT_DIR)


In [ ]:
from Methods.BaselineCNN import set_seed
from Methods.GroupAwareEvaluation import (
    evaluate_uni_intervention_faithfulness,
    evaluate_uni_intervention_stability,
    paired_attribution_method_comparison,
    paired_stability_method_comparison,
)
from Methods.UNIAttribution import build_uni_classifier, resolve_uni_transform

REFERENCE_SEED = 41
ALL_SEEDS = (11, 23, 41, 57, 73, 89, 101, 131, 151, 181)
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
UNI_MODEL_NAME = 'hf-hub:MahmoodLab/uni'
LOCAL_UNI_ASSETS_DIR = None

# The penultimate block leaves one downstream block in which a patched token
# can influence the final CLS token used by the frozen classifier.
INTERVENTION_LAYER = -2
REPLACEMENT = 'image_patch_mean'
INTERVENTION_BATCH_SIZE = 64
RANDOM_REPEATS = 20
BOOTSTRAP_ITERATIONS = 5000

RUN_FAITHFULNESS = True
RUN_STABILITY = True
set_seed(REFERENCE_SEED)
print('Device:', DEVICE)


## 1. Reuse the frozen cohort and checkpoints

The cohort file is read rather than rebuilt. Its SHA-256 digest is checked against the metadata written before attribution analysis. Absolute paths are rebased through the unchanged relative paths only when the original online path is unavailable.


In [ ]:
if not COHORT_PATH.is_file() or not COHORT_METADATA_PATH.is_file():
    raise FileNotFoundError('Run notebook 04 and freeze its cohort before this experiment')
cohort_metadata = json.loads(COHORT_METADATA_PATH.read_text())
cohort_digest = hashlib.sha256(COHORT_PATH.read_bytes()).hexdigest()
assert cohort_digest == cohort_metadata['sha256'], 'Frozen cohort digest changed'
cohort = pd.read_csv(COHORT_PATH)
missing_paths = ~cohort['path'].map(lambda value: Path(value).is_file())
cohort.loc[missing_paths, 'path'] = cohort.loc[missing_paths, 'relative_path'].map(
    lambda relative: str(DATASET_DIR / relative)
)
assert cohort['path'].map(lambda value: Path(value).is_file()).all()
assert cohort['cohort_id'].nunique() == cohort_metadata['image_count'] == len(cohort)
assert cohort['label'].nunique() == cohort_metadata['class_count'] == 8
assert cohort['case_id'].nunique() == cohort_metadata['source_group_count']
assert cohort['selected_without_attribution'].all()

required_checkpoints = [
    UNI_CHECKPOINT_DIR / f'seed_{seed}' / f'fold_{fold}.pt'
    for seed in ALL_SEEDS
    for fold in sorted(cohort['fold'].unique())
]
missing_checkpoints = [path for path in required_checkpoints if not path.is_file()]
if missing_checkpoints:
    raise FileNotFoundError(
        f'{len(missing_checkpoints)} required UNI OOF checkpoints are missing; '
        'complete notebook 04 before running intervention stability'
    )
display(pd.crosstab(cohort['class_name'], cohort['cohort_stratum']))
display(pd.crosstab(cohort['class_name'], cohort['case_id']))
print('Frozen cohort images:', len(cohort))
experiment_manifest = {
    'source_cohort_path': str(COHORT_PATH),
    'source_cohort_sha256': cohort_digest,
    'source_cohort_metadata': cohort_metadata,
    'reference_seed': REFERENCE_SEED,
    'stability_seeds': list(ALL_SEEDS),
    'classifier_checkpoints_reused': True,
    'data_split_changed': False,
    'classifier_training_changed': False,
    'cohort_selection_changed': False,
    'intervention_layer': INTERVENTION_LAYER,
    'replacement': REPLACEMENT,
}
(ARTIFACT_DIR / 'experiment_manifest.json').write_text(
    json.dumps(experiment_manifest, indent=2)
)


## 2. Load the unchanged frozen UNI classifier


In [ ]:
from huggingface_hub import login
login()

class_table = cohort[['class_name', 'label']].drop_duplicates().sort_values('label')
CLASS_NAMES = class_table['class_name'].tolist()
uni_model = build_uni_classifier(
    PROJECT_ROOT,
    num_classes=len(CLASS_NAMES),
    device=DEVICE,
    model_name=UNI_MODEL_NAME,
    assets_dir=LOCAL_UNI_ASSETS_DIR,
)
assert all(not parameter.requires_grad for parameter in uni_model.encoder.parameters())
uni_transform, uni_data_config = resolve_uni_transform(uni_model.encoder)
print('UNI blocks:', len(uni_model.encoder.blocks))
print('Intervention block index:', len(uni_model.encoder.blocks) + INTERVENTION_LAYER)
print('Preprocessing:', uni_data_config)


## 3. Intervention definition

For image (x), target class (y), and contextualized patch token (h_j^{(ell)}), the signed intervention score is

\[
\tau_j = z_y(x) - z_y\left(do\left(h_j^{(\ell)}=\tilde h^{(\ell)}\right)\right).
\]

- **Layer:** output of the penultimate UNI Transformer block. One final block plus the encoder norm remains, allowing the patched token to affect CLS.
- **Replacement:** the mean of the 196 contextualized patch tokens from the same image at that layer. This preserves activation scale while removing patch-specific information.
- **Independence:** each patch is replaced in a separate intervention; patches are not cumulatively altered.
- **Positive effect:** the original token supports the target logit relative to the replacement.
- **Negative effect:** the original token suppresses the target logit relative to the replacement.

These are effects on the trained model's prediction, not biological causal effects.


## 4. Paired faithfulness evaluation on the unchanged cohort


In [ ]:
if RUN_FAITHFULNESS:
    intervention_metrics, intervention_curves = evaluate_uni_intervention_faithfulness(
        uni_model,
        cohort,
        UNI_CHECKPOINT_DIR,
        REFERENCE_SEED,
        DEVICE,
        FAITHFULNESS_DIR,
        image_transform=uni_transform,
        intervention_layer=INTERVENTION_LAYER,
        replacement=REPLACEMENT,
        intervention_batch_size=INTERVENTION_BATCH_SIZE,
        random_repeats=RANDOM_REPEATS,
        heatmap_limit=32,
    )
else:
    intervention_metrics = pd.read_csv(
        FAITHFULNESS_DIR / 'uni_intervention_faithfulness_metrics.csv'
    )
    intervention_curves = pd.read_csv(
        FAITHFULNESS_DIR / 'uni_intervention_deletion_curves.csv'
    )

expected_methods = {'gradient_attention_rollout', 'intervention_activation_patch'}
assert set(intervention_metrics['method']) == expected_methods
assert intervention_metrics.groupby(['method', 'target_role'])['cohort_id'].nunique().unstack(0).nunique(axis=1).eq(1).all()
faithfulness_summary = (
    intervention_metrics.groupby(
        ['method', 'target_role', 'class_name', 'correct'], dropna=False
    )
    .agg(
        images=('cohort_id', 'nunique'),
        mean_spearman=('attribution_occlusion_spearman', 'mean'),
        mean_top_logit_auc=('top_target_logit_auc', 'mean'),
        mean_random_logit_auc=('random_target_logit_auc', 'mean'),
        mean_bottom_logit_auc=('bottom_target_logit_auc', 'mean'),
        mean_top_minus_random_logit_auc=('top_minus_random_target_logit_auc', 'mean'),
        mean_top_minus_random_margin_auc=('top_minus_random_margin_auc', 'mean'),
    )
    .reset_index()
)
faithfulness_summary.to_csv(FAITHFULNESS_DIR / 'aggregate_faithfulness_summary.csv', index=False)
display(
    intervention_metrics.groupby(['method', 'target_role'])
    .agg(
        images=('cohort_id', 'nunique'),
        mean_spearman=('attribution_occlusion_spearman', 'mean'),
        top_minus_random_logit_auc=('top_minus_random_target_logit_auc', 'mean'),
        top_minus_random_margin_auc=('top_minus_random_margin_auc', 'mean'),
    ).style.format(precision=4)
)


## 5. Paired Wilcoxon tests and hierarchical bootstrap intervals


In [ ]:
paired_tests, paired_values = paired_attribution_method_comparison(
    intervention_metrics,
    model_name='UNI',
    method_a='intervention_activation_patch',
    method_b='gradient_attention_rollout',
    bootstrap_iterations=BOOTSTRAP_ITERATIONS,
    confidence=0.95,
    random_seed=2027,
)
paired_tests.to_csv(FAITHFULNESS_DIR / 'paired_intervention_vs_gradient_tests.csv', index=False)
paired_values.to_csv(FAITHFULNESS_DIR / 'paired_intervention_vs_gradient_values.csv', index=False)
primary_tests = paired_tests.query(
    "comparison_target == 'true_class' and subgroup == 'all'"
).copy()
display(primary_tests.style.format(precision=4))


## 6. Cross-seed attribution stability


In [ ]:
if RUN_STABILITY:
    stability_predictions, stability_pairs, stability_images = (
        evaluate_uni_intervention_stability(
            uni_model,
            cohort,
            UNI_CHECKPOINT_DIR,
            ALL_SEEDS,
            DEVICE,
            STABILITY_DIR,
            image_transform=uni_transform,
            intervention_layer=INTERVENTION_LAYER,
            replacement=REPLACEMENT,
            intervention_batch_size=INTERVENTION_BATCH_SIZE,
        )
    )
else:
    stability_predictions = pd.read_csv(
        STABILITY_DIR / 'uni_intervention_stability_predictions.csv'
    )
    stability_pairs = pd.read_csv(
        STABILITY_DIR / 'uni_intervention_stability_seed_pairs.csv'
    )
    stability_images = pd.read_csv(
        STABILITY_DIR / 'uni_intervention_stability_per_image.csv'
    )

stability_summary = (
    stability_images.groupby(['method', 'class_name', 'cohort_stratum'])
    [[
        'pairwise_prediction_agreement',
        'mean_predicted_class_spearman_same_prediction',
        'mean_common_true_class_spearman',
    ]]
    .agg(['mean', 'sem'])
    .reset_index()
)
stability_summary.to_csv(STABILITY_DIR / 'aggregate_stability_summary.csv', index=False)
stability_tests, paired_stability_values = paired_stability_method_comparison(
    stability_images,
    method_a='intervention_activation_patch',
    method_b='gradient_attention_rollout',
    bootstrap_iterations=BOOTSTRAP_ITERATIONS,
    confidence=0.95,
    random_seed=2027,
)
stability_tests.to_csv(STABILITY_DIR / 'paired_stability_tests.csv', index=False)
paired_stability_values.to_csv(STABILITY_DIR / 'paired_stability_values.csv', index=False)
display(
    stability_images.groupby('method')[[
        'pairwise_prediction_agreement',
        'mean_predicted_class_spearman_same_prediction',
        'mean_common_true_class_spearman',
    ]].mean().style.format(precision=4)
)
display(stability_tests.query("subgroup == 'all'").style.format(precision=4))


## 7. Main comparison figures


In [ ]:
method_order = ['gradient_attention_rollout', 'intervention_activation_patch']
method_labels = {
    'gradient_attention_rollout': 'Gradient-weighted rollout',
    'intervention_activation_patch': 'Activation intervention',
}

true_metrics = intervention_metrics[
    intervention_metrics['target_class'] == intervention_metrics['true_class']
].copy()
true_metrics['method_label'] = true_metrics['method'].map(method_labels)
fig, axes = plt.subplots(1, 3, figsize=(17, 5), constrained_layout=True)
for axis, metric, title in zip(
    axes,
    [
        'attribution_occlusion_spearman',
        'top_minus_random_target_logit_auc',
        'top_minus_random_margin_auc',
    ],
    [
        'Attribution-input occlusion correlation',
        'Top minus random target-logit AUC',
        'Top minus random margin AUC',
    ],
):
    sns.boxplot(data=true_metrics, x='method_label', y=metric, ax=axis, showfliers=False)
    axis.axhline(0, color='black', linewidth=0.8, linestyle='--')
    axis.set(title=title, xlabel='')
    axis.tick_params(axis='x', rotation=15)
fig.savefig(FIGURE_DIR / '01_primary_faithfulness_comparison.png', dpi=240, bbox_inches='tight')
plt.show()

predicted_curves = intervention_curves[
    intervention_curves['target_role'] == 'predicted'
].copy()
predicted_curves['method_label'] = predicted_curves['method'].map(method_labels)
curve_summary = (
    predicted_curves.groupby(['method_label', 'strategy', 'fraction_removed'])
    [['target_class_logit_drop', 'target_margin_drop']]
    .agg(['mean', 'sem']).reset_index()
)
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
for axis, value, title in [
    (axes[0], 'target_class_logit_drop', 'Target-logit deletion'),
    (axes[1], 'target_margin_drop', 'Classification-margin deletion'),
]:
    for method in method_labels.values():
        for strategy, linestyle in [('top', '-'), ('random', '--'), ('bottom', ':')]:
            subset = predicted_curves[
                (predicted_curves['method_label'] == method)
                & (predicted_curves['strategy'] == strategy)
            ].groupby('fraction_removed')[value].mean().reset_index()
            axis.plot(subset['fraction_removed'], subset[value], linestyle=linestyle,
                      marker='o', label=f'{method}: {strategy}')
    axis.set(title=title, xlabel='Fraction of input patches deleted', ylabel='Score decrease')
axes[1].legend(fontsize=8, frameon=False, bbox_to_anchor=(1.02, 1), loc='upper left')
fig.savefig(FIGURE_DIR / '02_logit_margin_deletion_curves.png', dpi=240, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)
sns.pointplot(data=true_metrics, x='class_name', y='attribution_occlusion_spearman',
              hue='method_label', errorbar=('ci', 95), ax=axes[0])
axes[0].set(title='Per-class faithfulness', xlabel='', ylabel='Spearman correlation')
axes[0].tick_params(axis='x', rotation=55)
sns.boxplot(data=true_metrics, x='correct', y='attribution_occlusion_spearman',
            hue='method_label', showfliers=False, ax=axes[1])
axes[1].set(title='Correct versus incorrect predictions', xlabel='Correct prediction',
            ylabel='Spearman correlation')
fig.savefig(FIGURE_DIR / '03_class_and_error_faithfulness.png', dpi=240, bbox_inches='tight')
plt.show()

fig, axis = plt.subplots(figsize=(9, 5))
stability_plot = stability_images.copy()
stability_plot['method_label'] = stability_plot['method'].map(method_labels)
sns.boxplot(data=stability_plot, x='method_label',
            y='mean_common_true_class_spearman', showfliers=False, ax=axis)
axis.set(title='Attribution stability across classifier seeds', xlabel='',
         ylabel='Mean true-class map Spearman')
fig.tight_layout()
fig.savefig(FIGURE_DIR / '04_attribution_stability.png', dpi=240, bbox_inches='tight')
plt.show()


## 8. Prespecified decision gate for causal-ranking regularization


In [ ]:
# Stage 2 is justified only if the intervention ranking has a meaningful
# correlation gain and statistically supported deletion improvement.
primary = primary_tests.set_index('metric')
required_metrics = {
    'attribution_occlusion_spearman',
    'top_minus_random_target_logit_auc',
    'top_minus_random_margin_auc',
}
missing = required_metrics.difference(primary.index)
if missing:
    raise RuntimeError(f'Missing prespecified decision metrics: {sorted(missing)}')

spearman_row = primary.loc['attribution_occlusion_spearman']
logit_row = primary.loc['top_minus_random_target_logit_auc']
margin_row = primary.loc['top_minus_random_margin_auc']

def statistically_supported(row):
    return bool(
        row['wilcoxon_p_intervention_better_holm'] < 0.05
        and row['oriented_hierarchical_bootstrap_ci_low'] > 0
    )

meaningful_spearman_gain = bool(
    spearman_row['mean_oriented_improvement'] >= 0.05
    and statistically_supported(spearman_row)
)
deletion_support = statistically_supported(logit_row) or statistically_supported(margin_row)
causal_ranking_justified = bool(meaningful_spearman_gain and deletion_support)

decision = {
    'causal_ranking_regularization_justified': causal_ranking_justified,
    'training_loss_implemented_in_this_experiment': False,
    'required_mean_spearman_gain': 0.05,
    'observed_mean_spearman_gain': float(spearman_row['mean_oriented_improvement']),
    'spearman_statistically_supported': statistically_supported(spearman_row),
    'target_logit_deletion_statistically_supported': statistically_supported(logit_row),
    'margin_deletion_statistically_supported': statistically_supported(margin_row),
    'decision_rule': (
        'mean Spearman gain >= 0.05 with Holm-adjusted one-sided Wilcoxon p < 0.05 '
        'and positive source-group hierarchical CI, plus supported improvement in '
        'at least one top-minus-random deletion endpoint'
    ),
}
(ARTIFACT_DIR / 'causal_regularization_decision.json').write_text(
    json.dumps(decision, indent=2)
)
display(pd.Series(decision))

if causal_ranking_justified:
    stage2_spec = {
        'status': 'eligible_for_separate_preregistered_experiment',
        'teacher_target': 'signed intervention target-logit effects tau_j',
        'candidate_loss': 'pairwise logistic causal-ranking loss',
        'required_controls': [
            'unchanged grouped folds and cohort-independent training data',
            'lambda_causal selected without using the frozen faithfulness cohort',
            'classification and interpretability reported separately',
        ],
        'implemented_now': False,
    }
    (ARTIFACT_DIR / 'stage2_causal_ranking_experiment_spec.json').write_text(
        json.dumps(stage2_spec, indent=2)
    )
    print('Stage 1 supports preparing a separate causal-ranking experiment.')
else:
    print('Do not add a causal-ranking loss: the prespecified Stage 1 gate was not met.')

print('\nInterpretation guardrails:')
print('- Intervention scores are causal only with respect to the model computation.')
print('- Positive tau means support for the target score relative to the chosen replacement.')
print('- Negative tau means suppression of the target score relative to the replacement.')
print('- No biological causality claim is supported without external pathology annotations.')
